# 04 — ItemKNN / Sparse Item-Item Recommender

This notebook implements a standalone sparse item-item neighbourhood recommender for the Kaggle-style recommender-system challenge.

Why this model next:

- It is a strong independent architecture from LightGCN and SASRec.
- It is fast enough to tune several variants.
- It often ensembles well with graph embeddings because it captures direct item co-occurrence patterns.
- It does not depend on item metadata, so missing metadata is not a problem.

The output format matches the previous notebooks: each run writes to a unique `outputs/itemknn_<run_id>/` directory and creates a `submission_itemknn.csv` file with the required `ID,user_id,item_id` columns.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import uuid
import json
import time
import math
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import scipy.sparse as sp

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = Path('data/')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:8]
OUTPUT_DIR = DATA_DIR / 'outputs' / f'itemknn_{RUN_ID}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Output directory:', OUTPUT_DIR)

Output directory: data/outputs/itemknn_20260610_181959_ba696400


In [2]:
# Core configuration.
# This default is a strong and fast raw co-occurrence ItemKNN baseline.
# Use RUN_GRID_SEARCH to test a compact set of variants on the temporal validation split.
FAST_DEV_RUN = False
RUN_VALIDATION = True
RUN_GRID_SEARCH = True
RUN_FINAL_TRAINING = True
USE_BEST_VALIDATION_CONFIG_FOR_FINAL = True

CONFIG = {
    'similarity': 'cooccurrence',   # one of: cooccurrence, cosine, bm25, bm25_cosine, asymmetric
    'topk': 500,                    # keep top-k neighbours per source item
    'shrink': 0.0,                  # useful for cosine/asymmetric variants
    'asym_alpha': 0.5,              # only used for asymmetric similarity
    'bm25_k1': 100.0,               # only used for bm25 variants
    'bm25_b': 0.8,                  # only used for bm25 variants
    'popularity_blend_alpha': 0.03, # add small global popularity score at inference
    'recent_popularity_blend_alpha': 0.00,
    'recent_days': 180,
    'recommend_batch_size': 512,
}

# Compact validation grid. It should run quickly on this dataset.
GRID = [
    {'similarity': 'cooccurrence', 'topk': 100, 'shrink': 0.0,   'popularity_blend_alpha': 0.00},
    {'similarity': 'cooccurrence', 'topk': 200, 'shrink': 0.0,   'popularity_blend_alpha': 0.03},
    {'similarity': 'cooccurrence', 'topk': 500, 'shrink': 0.0,   'popularity_blend_alpha': 0.03},
    {'similarity': 'cooccurrence', 'topk': 500, 'shrink': 0.0,   'popularity_blend_alpha': 0.08},
    {'similarity': 'cosine',       'topk': 200, 'shrink': 100.0, 'popularity_blend_alpha': 0.03},
    {'similarity': 'bm25',         'topk': 200, 'shrink': 0.0,   'popularity_blend_alpha': 0.03},
    {'similarity': 'asymmetric',   'topk': 200, 'shrink': 50.0,  'popularity_blend_alpha': 0.03, 'asym_alpha': 0.3},
]

if FAST_DEV_RUN:
    GRID = GRID[:2]
    RUN_FINAL_TRAINING = False

with open(OUTPUT_DIR / 'itemknn_initial_config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)

In [3]:
# Load data and create the same temporal split as the previous notebooks.
train_raw = pd.read_csv(DATA_DIR / 'train.csv')
test_raw = pd.read_csv(DATA_DIR / 'test.csv')
sample = pd.read_csv(DATA_DIR / 'sample_submission.csv')

# Exact duplicate events are removed. In this dataset, repeated user-item pairs have the same timestamp.
train_all = (
    train_raw
    .sort_values(['timestamp', 'user_id', 'item_id'])
    .drop_duplicates(['user_id', 'item_id'], keep='first')
    .reset_index(drop=True)
)

test_min_ts = test_raw['timestamp'].min()
train_fit = train_all[train_all['timestamp'] < test_min_ts].reset_index(drop=True)
valid = train_all[train_all['timestamp'] >= test_min_ts].reset_index(drop=True)

if FAST_DEV_RUN:
    # Keep enough users/items for a smoke test but do not use this for real validation.
    keep_users = train_fit['user_id'].drop_duplicates().sample(3000, random_state=SEED)
    train_fit = train_fit[train_fit['user_id'].isin(keep_users)].reset_index(drop=True)
    valid = valid[valid['user_id'].isin(keep_users)].reset_index(drop=True)

print('train_raw:', train_raw.shape)
print('train_all deduped:', train_all.shape)
print('train_fit:', train_fit.shape)
print('valid:', valid.shape)
print('sample:', sample.shape)
print('validation users:', valid['user_id'].nunique())
print('sample users in validation:', valid[valid['user_id'].isin(sample['user_id'])]['user_id'].nunique())

train_raw: (162727, 3)
train_all deduped: (158471, 3)
train_fit: (143313, 3)
valid: (15158, 3)
sample: (2255, 3)
validation users: 7437
sample users in validation: 1285


In [4]:
def make_targets(df):
    return df.groupby('user_id')['item_id'].apply(lambda x: set(map(int, x))).to_dict()


def recall_ndcg_at_k(recs, targets, k=10):
    recalls = []
    ndcgs = []
    for user_id, true_items in targets.items():
        pred = recs.get(user_id, [])[:k]
        if not true_items:
            continue
        hits = [1 if item in true_items else 0 for item in pred]
        recall = sum(hits) / min(k, len(true_items))
        dcg = sum(hit / math.log2(i + 2) for i, hit in enumerate(hits))
        ideal_hits = min(k, len(true_items))
        idcg = sum(1.0 / math.log2(i + 2) for i in range(ideal_hits))
        recalls.append(recall)
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)
    return float(np.mean(recalls)), float(np.mean(ndcgs))


def write_submission(recs, sample_df, path):
    out = sample_df[['ID', 'user_id']].copy()
    out['item_id'] = out['user_id'].map(lambda u: ','.join(map(str, recs[int(u)][:10])))
    lens = out['item_id'].str.split(',').map(len)
    assert lens.eq(10).all(), 'Every row must contain exactly 10 comma-separated item IDs.'
    out.to_csv(path, index=False)
    return out


def recommendation_diagnostics(submission_df):
    strings = submission_df['item_id'].astype(str)
    all_items = [int(x) for row in strings for x in row.split(',')]
    print('Submission rows:', len(submission_df))
    print('Unique recommendation strings:', strings.nunique())
    print('Unique recommended items:', len(set(all_items)))
    print('Most common recommendation strings:')
    display(strings.value_counts().head(5).to_frame('count'))

In [5]:
@dataclass
class ItemKNNData:
    interactions: pd.DataFrame
    X: sp.csr_matrix
    raw_users: np.ndarray
    raw_items: np.ndarray
    user2idx: dict
    item2idx: dict
    idx2item: np.ndarray
    user_seen_items: dict
    item_popularity: np.ndarray
    global_pop_score: np.ndarray
    recent_pop_score: np.ndarray


def build_itemknn_data(interactions: pd.DataFrame, recent_days=180) -> ItemKNNData:
    interactions = interactions.copy()
    raw_users = np.sort(interactions['user_id'].unique())
    raw_items = np.sort(interactions['item_id'].unique())
    user2idx = {int(u): i for i, u in enumerate(raw_users)}
    item2idx = {int(it): i for i, it in enumerate(raw_items)}
    idx2item = raw_items.astype(int)

    row = interactions['user_id'].map(user2idx).to_numpy()
    col = interactions['item_id'].map(item2idx).to_numpy()
    data = np.ones(len(interactions), dtype=np.float32)
    X = sp.csr_matrix((data, (row, col)), shape=(len(raw_users), len(raw_items)), dtype=np.float32)
    X.sum_duplicates()
    X.data[:] = 1.0

    user_seen_items = interactions.groupby('user_id')['item_id'].apply(lambda s: set(map(int, s))).to_dict()
    item_popularity = np.asarray(X.sum(axis=0)).ravel().astype(np.float32)
    global_pop_score = item_popularity / (item_popularity.max() + 1e-12)

    max_ts = interactions['timestamp'].max()
    window_start = max_ts - int(recent_days * 24 * 3600 * 1000)
    recent = interactions[interactions['timestamp'] >= window_start]
    if len(recent):
        recent_col = recent['item_id'].map(item2idx).to_numpy()
        recent_counts = np.bincount(recent_col, minlength=len(raw_items)).astype(np.float32)
    else:
        recent_counts = np.zeros(len(raw_items), dtype=np.float32)
    recent_pop_score = recent_counts / (recent_counts.max() + 1e-12)

    return ItemKNNData(
        interactions=interactions,
        X=X,
        raw_users=raw_users,
        raw_items=raw_items,
        user2idx=user2idx,
        item2idx=item2idx,
        idx2item=idx2item,
        user_seen_items=user_seen_items,
        item_popularity=item_popularity,
        global_pop_score=global_pop_score,
        recent_pop_score=recent_pop_score,
    )


def topk_filter_rows(matrix: sp.spmatrix, topk: int) -> sp.csr_matrix:
    # Keep only the top-k values in each row of a sparse matrix.
    matrix = matrix.tocsr()
    data = []
    indices = []
    indptr = [0]

    for row_id in range(matrix.shape[0]):
        start, end = matrix.indptr[row_id], matrix.indptr[row_id + 1]
        row_data = matrix.data[start:end]
        row_indices = matrix.indices[start:end]

        if row_data.size > topk:
            keep = np.argpartition(row_data, -topk)[-topk:]
            row_data = row_data[keep]
            row_indices = row_indices[keep]

        if row_data.size:
            order = np.argsort(row_data)[::-1]
            row_data = row_data[order]
            row_indices = row_indices[order]

        data.extend(row_data.tolist())
        indices.extend(row_indices.tolist())
        indptr.append(len(data))

    return sp.csr_matrix(
        (np.asarray(data, dtype=np.float32), np.asarray(indices, dtype=np.int32), np.asarray(indptr, dtype=np.int64)),
        shape=matrix.shape,
    )

In [6]:
def bm25_weight(X: sp.csr_matrix, k1=100.0, b=0.8) -> sp.csr_matrix:
    # BM25 reweighting for implicit user-item matrices.
    # Rows are users/documents, columns are items/terms.
    X = X.tocsr(copy=True).astype(np.float32)
    n_users, n_items = X.shape

    item_df = np.diff(X.tocsc().indptr).astype(np.float32)
    idf = np.log((n_users - item_df + 0.5) / (item_df + 0.5))
    idf = np.maximum(idf, 0.0).astype(np.float32)

    user_len = np.asarray(X.sum(axis=1)).ravel().astype(np.float32)
    avg_len = max(float(user_len.mean()), 1e-12)

    X = X.tocoo(copy=True)
    denom = X.data + k1 * (1.0 - b + b * user_len[X.row] / avg_len)
    X.data = X.data * (k1 + 1.0) / denom * idf[X.col]
    return X.tocsr()


def compute_item_similarity(data: ItemKNNData, config: dict) -> sp.csr_matrix:
    # Train the item-item neighbourhood model.
    X = data.X.astype(np.float32)
    similarity = config.get('similarity', 'cooccurrence')
    topk = int(config.get('topk', 200))
    shrink = float(config.get('shrink', 0.0))

    if similarity in {'bm25', 'bm25_cosine'}:
        Xw = bm25_weight(X, k1=float(config.get('bm25_k1', 100.0)), b=float(config.get('bm25_b', 0.8)))
    else:
        Xw = X

    W = (Xw.T @ Xw).tocsr().astype(np.float32)
    W.setdiag(0.0)
    W.eliminate_zeros()

    if similarity in {'cosine', 'bm25_cosine'}:
        norms = np.sqrt(np.asarray(Xw.power(2).sum(axis=0)).ravel()).astype(np.float32) + 1e-12
        W = W.tocoo(copy=True)
        W.data = W.data / (norms[W.row] * norms[W.col] + shrink)
        W = W.tocsr()

    elif similarity == 'asymmetric':
        # Asymmetric cosine: sim(i,j) = cooc(i,j) / (deg_i^alpha * deg_j^(1-alpha) + shrink)
        alpha = float(config.get('asym_alpha', 0.5))
        deg = np.asarray(X.sum(axis=0)).ravel().astype(np.float32) + 1e-12
        W = W.tocoo(copy=True)
        denom = (deg[W.row] ** alpha) * (deg[W.col] ** (1.0 - alpha)) + shrink
        W.data = W.data / denom
        W = W.tocsr()

    elif similarity == 'cooccurrence':
        if shrink > 0:
            W.data = W.data / (W.data + shrink)

    elif similarity == 'bm25':
        if shrink > 0:
            W.data = W.data / (W.data + shrink)

    else:
        raise ValueError(f'Unknown similarity type: {similarity}')

    W = topk_filter_rows(W, topk=topk)
    W.eliminate_zeros()
    return W

In [7]:
class ItemKNNRecommender:
    def __init__(self, data: ItemKNNData, similarity_matrix: sp.csr_matrix, config: dict):
        self.data = data
        self.W = similarity_matrix.tocsr().astype(np.float32)
        self.config = dict(config)
        self.fallback_items = data.idx2item[np.argsort(-data.item_popularity)].astype(int).tolist()

    def recommend(self, raw_users, k=10, batch_size=512):
        raw_users = [int(u) for u in raw_users]
        recs = {}
        n_items = self.data.X.shape[1]
        pop_alpha = float(self.config.get('popularity_blend_alpha', 0.0))
        recent_alpha = float(self.config.get('recent_popularity_blend_alpha', 0.0))

        for start in range(0, len(raw_users), batch_size):
            batch_users = raw_users[start:start + batch_size]
            mapped = [self.data.user2idx.get(u, -1) for u in batch_users]
            known_user_indices = [idx for idx in mapped if idx >= 0]

            scores = np.zeros((len(batch_users), n_items), dtype=np.float32)
            if known_user_indices:
                known_scores = self.data.X[known_user_indices].dot(self.W).toarray().astype(np.float32)
                ptr = 0
                for row_id, user_idx in enumerate(mapped):
                    if user_idx >= 0:
                        scores[row_id] = known_scores[ptr]
                        ptr += 1

            if pop_alpha > 0:
                scores += pop_alpha * self.data.global_pop_score[None, :]
            if recent_alpha > 0:
                scores += recent_alpha * self.data.recent_pop_score[None, :]

            for row_id, raw_user in enumerate(batch_users):
                seen_items = self.data.user_seen_items.get(raw_user, set())
                if seen_items:
                    seen_idx = [self.data.item2idx[item] for item in seen_items if item in self.data.item2idx]
                    scores[row_id, seen_idx] = -np.inf

                row_scores = scores[row_id]
                if np.all(~np.isfinite(row_scores)):
                    chosen = []
                else:
                    n_select = min(max(k * 5, k), n_items)
                    top_idx = np.argpartition(row_scores, -n_select)[-n_select:]
                    top_idx = top_idx[np.argsort(row_scores[top_idx])[::-1]]
                    chosen = [int(self.data.idx2item[i]) for i in top_idx if np.isfinite(row_scores[i])]

                if len(chosen) < k:
                    chosen_set = set(chosen)
                    chosen += [
                        item for item in self.fallback_items
                        if item not in chosen_set and item not in seen_items
                    ][:k - len(chosen)]

                recs[raw_user] = chosen[:k]
        return recs


def train_itemknn(train_df, config):
    t0 = time.time()
    data = build_itemknn_data(train_df, recent_days=int(config.get('recent_days', 180)))
    W = compute_item_similarity(data, config)
    elapsed = time.time() - t0
    print(f"Trained ItemKNN: users={data.X.shape[0]:,}, items={data.X.shape[1]:,}, interactions={data.X.nnz:,}, W.nnz={W.nnz:,}, time={elapsed:.2f}s")
    return ItemKNNRecommender(data, W, config)

In [8]:
def evaluate_config(config, train_df, valid_df, sample_users=None):
    model = train_itemknn(train_df, config)
    valid_targets = make_targets(valid_df)

    recs_all = model.recommend(
        raw_users=list(valid_targets.keys()),
        k=10,
        batch_size=int(config.get('recommend_batch_size', 512)),
    )
    recall_all, ndcg_all = recall_ndcg_at_k(recs_all, valid_targets, k=10)

    recall_sample = np.nan
    ndcg_sample = np.nan
    n_sample_valid_users = 0
    if sample_users is not None:
        sample_users = set(map(int, sample_users))
        sample_targets = {u: items for u, items in valid_targets.items() if u in sample_users}
        n_sample_valid_users = len(sample_targets)
        if sample_targets:
            recs_sample = {u: recs_all[u] for u in sample_targets.keys()}
            recall_sample, ndcg_sample = recall_ndcg_at_k(recs_sample, sample_targets, k=10)

    result = dict(config)
    result.update({
        'valid_recall10_all_users': recall_all,
        'valid_ndcg10_all_users': ndcg_all,
        'valid_recall10_sample_users': recall_sample,
        'valid_ndcg10_sample_users': ndcg_sample,
        'n_valid_users': len(valid_targets),
        'n_sample_valid_users': n_sample_valid_users,
    })
    return result, model

validation_results = []
best_config = dict(CONFIG)
best_model = None

if RUN_VALIDATION:
    configs_to_try = []
    if RUN_GRID_SEARCH:
        for override in GRID:
            cfg = dict(CONFIG)
            cfg.update(override)
            configs_to_try.append(cfg)
    else:
        configs_to_try = [dict(CONFIG)]

    for i, cfg in enumerate(configs_to_try, start=1):
        print()
        print('=' * 90)
        print(f'Validation config {i}/{len(configs_to_try)}')
        print(json.dumps(cfg, indent=2))
        result, model = evaluate_config(cfg, train_fit, valid, sample_users=sample['user_id'])
        validation_results.append(result)
        print('Result:', json.dumps({k: v for k, v in result.items() if k.startswith('valid_') or k.startswith('n_')}, indent=2))

    validation_history = pd.DataFrame(validation_results)
    validation_history.to_csv(OUTPUT_DIR / 'itemknn_validation_history.csv', index=False)
    display(validation_history.sort_values('valid_recall10_sample_users', ascending=False))

    # The leaderboard only asks for sample_submission users, so choose by sample-user validation if available.
    score_col = 'valid_recall10_sample_users'
    if validation_history[score_col].isna().all():
        score_col = 'valid_recall10_all_users'
    best_row = validation_history.sort_values(score_col, ascending=False).iloc[0].to_dict()
    best_config = dict(CONFIG)
    for key in CONFIG.keys():
        if key in best_row and not pd.isna(best_row[key]):
            best_config[key] = best_row[key]
    # GRID can include keys not in CONFIG, e.g. asym_alpha.
    for key in ['asym_alpha']:
        if key in best_row and not pd.isna(best_row[key]):
            best_config[key] = best_row[key]
    print()
    print('Best config selected by', score_col)
    print(json.dumps(best_config, indent=2))
else:
    print('Skipping validation; using CONFIG for final training.')

with open(OUTPUT_DIR / 'itemknn_best_config.json', 'w') as f:
    json.dump(best_config, f, indent=2)


Validation config 1/7
{
  "similarity": "cooccurrence",
  "topk": 100,
  "shrink": 0.0,
  "asym_alpha": 0.5,
  "bm25_k1": 100.0,
  "bm25_b": 0.8,
  "popularity_blend_alpha": 0.0,
  "recent_popularity_blend_alpha": 0.0,
  "recent_days": 180,
  "recommend_batch_size": 512
}
Trained ItemKNN: users=23,284, items=13,441, interactions=143,313, W.nnz=754,848, time=0.26s
Result: {
  "valid_recall10_all_users": 0.013072133967656354,
  "valid_ndcg10_all_users": 0.008279123220423274,
  "valid_recall10_sample_users": 0.012381878821567537,
  "valid_ndcg10_sample_users": 0.00800510162846808,
  "n_valid_users": 7437,
  "n_sample_valid_users": 1285
}

Validation config 2/7
{
  "similarity": "cooccurrence",
  "topk": 200,
  "shrink": 0.0,
  "asym_alpha": 0.5,
  "bm25_k1": 100.0,
  "bm25_b": 0.8,
  "popularity_blend_alpha": 0.03,
  "recent_popularity_blend_alpha": 0.0,
  "recent_days": 180,
  "recommend_batch_size": 512
}
Trained ItemKNN: users=23,284, items=13,441, interactions=143,313, W.nnz=920,292,

,similarity,topk,shrink,asym_alpha,bm25_k1,bm25_b,popularity_blend_alpha,recent_popularity_blend_alpha,recent_days,recommend_batch_size,valid_recall10_all_users,valid_ndcg10_all_users,valid_recall10_sample_users,valid_ndcg10_sample_users,n_valid_users,n_sample_valid_users
4,cosine,200,100.0,0.5,100.0,0.8,0.03,0.0,180,512,0.013614,0.008675,0.014619,0.008750,7437,1285
0,cooccurrence,100,0.0,0.5,100.0,0.8,0.00,0.0,180,512,0.013072,0.008279,0.012382,0.008005,7437,1285
2,cooccurrence,500,0.0,0.5,100.0,0.8,0.03,0.0,180,512,0.014076,0.008835,0.012076,0.007872,7437,1285
3,cooccurrence,500,0.0,0.5,100.0,0.8,0.08,0.0,180,512,0.014076,0.008835,0.012076,0.007872,7437,1285
1,cooccurrence,200,0.0,0.5,100.0,0.8,0.03,0.0,180,512,0.013921,0.008776,0.012024,0.007778,7437,1285
6,asymmetric,200,50.0,0.3,100.0,0.8,0.03,0.0,180,512,0.011336,0.007173,0.009813,0.006203,7437,1285
5,bm25,200,0.0,0.5,100.0,0.8,0.03,0.0,180,512,0.010268,0.006741,0.007030,0.004861,7437,1285



Best config selected by valid_recall10_sample_users
{
  "similarity": "cosine",
  "topk": 200,
  "shrink": 100.0,
  "asym_alpha": 0.5,
  "bm25_k1": 100.0,
  "bm25_b": 0.8,
  "popularity_blend_alpha": 0.03,
  "recent_popularity_blend_alpha": 0.0,
  "recent_days": 180,
  "recommend_batch_size": 512
}


In [9]:
# Train on all deduplicated interactions and create final submission.
if RUN_FINAL_TRAINING:
    final_config = best_config if USE_BEST_VALIDATION_CONFIG_FOR_FINAL else CONFIG
    print('Final training config:')
    print(json.dumps(final_config, indent=2))

    final_model = train_itemknn(train_all, final_config)
    final_recs = final_model.recommend(
        raw_users=sample['user_id'].astype(int).tolist(),
        k=10,
        batch_size=int(final_config.get('recommend_batch_size', 512)),
    )

    submission_path = OUTPUT_DIR / 'submission_itemknn.csv'
    submission = write_submission(final_recs, sample, submission_path)

    # Convenience copy in /mnt/data, matching the style of previous generated submissions.
    root_submission_path = DATA_DIR / 'submission_itemknn.csv'
    submission.to_csv(root_submission_path, index=False)

    display(submission.head())
    print('Saved:', submission_path)
    print('Saved convenience copy:', root_submission_path)
    recommendation_diagnostics(submission)
else:
    print('RUN_FINAL_TRAINING=False; no final submission generated.')

Final training config:
{
  "similarity": "cosine",
  "topk": 200,
  "shrink": 100.0,
  "asym_alpha": 0.5,
  "bm25_k1": 100.0,
  "bm25_b": 0.8,
  "popularity_blend_alpha": 0.03,
  "recent_popularity_blend_alpha": 0.0,
  "recent_days": 180,
  "recommend_batch_size": 512
}
Trained ItemKNN: users=23,284, items=13,441, interactions=158,471, W.nnz=1,065,455, time=0.24s


,ID,user_id,item_id
0,12,12,"1290,2140,5303,9799,6971,9668,10608,7120,7146,..."
1,14,14,"7637,9668,5342,7713,11733,9160,2140,2847,12663..."
2,17,17,"2870,6971,1290,6571,12413,11110,11493,2085,814..."
3,21,21,"8421,2140,6971,1912,4798,479,7637,1634,11733,2218"
4,44,44,"5135,7120,9962,789,1290,4929,5008,1829,3697,9405"


Saved: data/outputs/itemknn_20260610_181959_ba696400/submission_itemknn.csv
Saved convenience copy: data/submission_itemknn.csv
Submission rows: 2255
Unique recommendation strings: 2241
Unique recommended items: 2639
Most common recommendation strings:


,count
item_id,
"7120,10663,4929,6243,12497,2916,8567,296,4777,13101",6
"1290,10663,7120,2140,12026,13147,7417,8633,13296,3697",3
"2870,6971,1290,6571,12413,11110,11493,2085,8148,2860",2
"10663,1290,7120,6243,2916,8567,4777,12497,296,3697",2
"5025,1290,6433,3789,1681,6220,9234,717,6971,2140",2


## Notes and next experiments

Recommended next steps after this notebook:

1. Submit the best ItemKNN variant if validation looks competitive.
2. Compare overlap with LightGCN and SASRec recommendations. Low overlap is good for ensembling.
3. Try rank averaging: `LightGCN + ItemKNN` first, then add SASRec if it improves validation.
4. If ItemKNN is strong, implement a related graph random-walk model next: RP3beta/P3alpha.
5. After several standalone recommenders are available, use their scores/ranks as features in a metadata-aware reranker.